# Profiling GPTQ INT4 LLM Inference: Diagnosing a Marlin Decode Bottleneck on NVIDIA T4

## Project Objective

This notebook investigates the GPU execution path of a GPTQ INT4 language model running with vLLM and Marlin.

The goal is not simply to benchmark INT4 inference. The objective is to identify where steady-state decode time is spent, determine why the dominant quantized kernels are limited, and use profiler evidence to guide a targeted optimization.

The workflow is:

**quantized inference → system-level profiling → kernel identification → hardware-counter analysis → bottleneck diagnosis → optimization experiment → validation**

### Test Configuration

- **Model:** `JunHowie/Qwen3-0.6B-GPTQ-Int4`
- **Runtime:** vLLM 0.29.0
- **GPU:** NVIDIA Tesla T4
- **Quantization:** GPTQ INT4
- **Primary kernel backend:** Marlin
- **Profilers:** NVIDIA Nsight Systems and Nsight Compute

The primary workload uses a short prompt and longer generation to emphasize autoregressive decode.

## 1. Environment Setup

Install vLLM and verify the CUDA/PyTorch environment before profiling.

The exact GPU and software versions matter because kernel selection and performance characteristics are hardware- and runtime-dependent.

In [ ]:
!pip install -U "vllm[bench]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 2

In [ ]:
import torch
import vllm

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))
print("vLLM:", vllm.__version__)

torch: 2.13.0+cu130
CUDA: 13.0
GPU: Tesla T4
capability: (7, 5)
vLLM: 0.29.0


### Environment Compatibility Cleanup

`torchaudio` is not required for this experiment and is removed to avoid dependency conflicts with the PyTorch/vLLM environment.

In [ ]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


## 2. Sanity Check: Verify the Quantized Inference Path

Before profiling, run a small inference workload to confirm that:

1. the GPTQ model loads successfully,
2. inference works on the T4,
3. vLLM recognizes the model as GPTQ INT4,
4. the expected quantized execution backend is selected.

During initialization, the important runtime message is:

`Using MarlinLinearKernel for AutoGPTQLinearMethod`

This confirms that the GPTQ linear layers are being executed through the Marlin quantized GEMM path rather than a generic dense FP16 implementation.

This run is only a functional sanity check; its latency is not used as a final performance result.

In [ ]:
!vllm bench latency \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --input-len 32 \
  --output-len 32 \
  --batch-size 1 \
  --num-iters-warmup 2 \
  --num-iters 1

INFO 09-20 16:08:19 [api_utils.py:286] non-default args: {'enable_prefix_caching': False, 'mm_device_do_normalize': None, 'enable_lora': None, 'reasoning_parser_plugin': '', 'enable_bf16x3_router_gemm': False, 'model': 'JunHowie/Qwen3-0.6B-GPTQ-Int4'}
config.json: 100% 1.92k/1.92k [00:00<00:00, 2.81MB/s]
INFO 09-20 16:08:36 [model.py:684] Resolved architecture: Qwen3ForCausalLM
WARNING 09-20 16:08:36 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-20 16:08:36 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:08:36 [model.py:2021] Using max model len 40960
INFO 09-20 16:08:37 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:08:38 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
tokenizer_config.json: 100% 5.42k/5.42k [00

## 3. Profiling Tools

Two NVIDIA profiling tools are used for different purposes:

- **Nsight Systems (`nsys`)** — identifies which kernels execute, how frequently they execute, and where GPU time is spent.
- **Nsight Compute (`ncu`)** — examines hardware-level behavior of an individual kernel, including occupancy, registers, shared memory, scheduler behavior, and compute/memory utilization.

The intended workflow is:

**Nsight Systems → identify expensive kernel → Nsight Compute → explain why it is expensive**

In [ ]:
!which nsys
!nsys --version

!which ncu
!ncu --version

/bin/bash: line 1: nsys: command not found
/usr/local/cuda/bin/ncu
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


In [ ]:
!apt-get update -qq
!apt-get install -y --no-install-recommends gnupg2 wget ca-certificates

!wget -qO- https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub \
  | gpg --dearmor \
  | tee /usr/share/keyrings/nvidia-devtools-keyring.gpg > /dev/null

!echo "deb [signed-by=/usr/share/keyrings/nvidia-devtools-keyring.gpg] \
https://developer.download.nvidia.com/devtools/repos/ubuntu$(source /etc/lsb-release; echo $DISTRIB_RELEASE | tr -d .)/$(dpkg --print-architecture)/ /" \
  | tee /etc/apt/sources.list.d/nvidia-devtools.list

!apt-get update -qq
!apt-get install -y nsight-systems-cli

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gnupg2 is already the newest version (2.4.4-2ubuntu17.6).
wget is already the newest version (1.21.4-1ubuntu4.5).
ca-certificates is already the newest version (20260601~24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 93 not upgraded.
deb [signed-by=/usr/share/keyrings/nvidia-devtools-keyring.gpg]  https://developer.download.nvidia.com/devtools/repos/ubuntu2404/amd64/ /
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted

In [ ]:
!nsys status -e

NVIDIA Nsight Systems version 2026.5.1.161-265138896106v0

General Check
- Platform: Linux
- Timestamp counter supported: Yes

CPU Profiling Environment Check
- Root privilege: enabled
- Linux Kernel Paranoid Level: 2 (some features maybe not available)
- Linux Distribution: Ubuntu
- Linux Kernel Version (6.6.122+): OK
- Linux perf_event_open syscall available: OK
- Sampling trigger event available: OK
- Intel(c) Last Branch Record support: Not Available
- CPU Profiling Environment (process-tree): OK
- CPU Profiling Environment (system-wide): OK

See the product documentation at https://docs.nvidia.com/nsight-systems for more information,
including information on how to set the Linux Kernel Paranoid Level.


# Part 1 — Profiling and Bottleneck Diagnosis

## 4. Initial System-Level Profile with Nsight Systems

The first profiling pass uses Nsight Systems to determine which CUDA kernel families dominate a decode-heavy workload.

### Workload

- Input length: 32 tokens
- Output length: 256 tokens
- Batch size: 1

The short prompt minimizes prefill relative to generation, while the 256-token output creates many autoregressive decode iterations.

### Important Methodology Note

This first trace profiles the entire `vllm bench latency` process. It therefore includes more than steady-state inference:

- model/runtime initialization,
- kernel warmup,
- compilation,
- CUDA Graph preparation,
- and the measured inference iteration.

The trace is useful for discovering kernel families, but its aggregate timing percentages should **not yet be treated as steady-state decode measurements**.

In [ ]:
!nsys profile \
  --trace=cuda,nvtx,osrt \
  --trace-fork-before-exec=true \
  --cuda-graph-trace=node \
  --force-overwrite=true \
  -o /content/decode_profile \
  vllm bench latency \
    --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
    --input-len 32 \
    --output-len 256 \
    --batch-size 1 \
    --num-iters-warmup 5 \
    --num-iters 1

INFO 09-20 16:11:36 [api_utils.py:286] non-default args: {'enable_prefix_caching': False, 'mm_device_do_normalize': None, 'enable_lora': None, 'reasoning_parser_plugin': '', 'enable_bf16x3_router_gemm': False, 'model': 'JunHowie/Qwen3-0.6B-GPTQ-Int4'}
INFO 09-20 16:11:36 [model.py:684] Resolved architecture: Qwen3ForCausalLM
WARNING 09-20 16:11:36 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-20 16:11:36 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:11:36 [model.py:2021] Using max model len 40960
INFO 09-20 16:11:37 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:11:38 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=4862) INFO 09-20 16:12:04 [core.py:123] Initializing a V1 LLM engine (v0.29.0) wi

In [ ]:
!nsys stats \
  --report cuda_gpu_kern_sum \
  --report cuda_api_sum \
  /content/decode_profile.nsys-rep

Generating SQLite file /content/decode_profile.sqlite from /content/decode_profile.nsys-rep
Processing [/content/decode_profile.sqlite] with [/opt/nvidia/nsight-systems-cli/2026.5.1/target-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)                                                  Name                                                
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  ----------------------------------------------------------------------------------------------------
     36.1    4,511,655,106     49,868      90,471.9      26,911.0      11,808   1,887,762    206,941.4  kernel_unified_attention                                                                            
     17.2    2,144,837,749      1,537   1,395,470.2   1,383,262.0   1,377,310   2,276,936    103,447.

## 5. Correcting the Measurement Boundary: Isolating Steady-State Decode

The initial trace revealed the relevant kernel families, but it also contained initialization, compilation, and CUDA Graph capture.

To obtain a meaningful decode profile, the measurement boundary is changed.

The corrected procedure is:

1. Start the vLLM server under Nsight Systems.
2. Allow model initialization and runtime setup to complete.
3. Send an unprofiled warm-up request.
4. Start profiling only for a subsequent request.
5. Capture one 32-token input / 256-token output request.
6. Stop the server and analyze only the captured range.

`--capture-range=cudaProfilerApi` is used so that Nsight Systems records the region explicitly triggered by vLLM's profiling interface.

This separates **steady-state inference** from one-time startup costs.

In [ ]:
%%bash

rm -f /content/steady_decode*

nohup nsys profile \
  --trace=cuda,nvtx,osrt \
  --trace-fork-before-exec=true \
  --cuda-graph-trace=node \
  --capture-range=cudaProfilerApi \
  --capture-range-end=repeat \
  --force-overwrite=true \
  -o /content/steady_decode \
  vllm serve JunHowie/Qwen3-0.6B-GPTQ-Int4 \
    --port 8000 \
    --profiler-config.profiler cuda \
  > /content/vllm_server.log 2>&1 &

echo $!

6270


In [ ]:
import time
import requests

for _ in range(180):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server ready")
            break
    except:
        pass

    time.sleep(1)

Server ready


In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 256 \
  --num-prompts 1 \
  --ignore-eos \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7a0b0479cd60>, trust_remote_code=False, seed=0, num_prompts=1, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=

### 5.1 Warm-Up Request — Not Profiled

The following request is intentionally executed without `--profile`.

Its purpose is to ensure that the model and runtime have already exercised the relevant inference path before measurement. The latency from this request is not used in the analysis.

In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 256 \
  --num-prompts 1 \
  --ignore-eos \
  --profile \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7d67000b0e00>, trust_remote_code=False, seed=0, num_prompts=1, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=

In [ ]:
!pkill -INT -f "vllm serve JunHowie/Qwen3-0.6B-GPTQ-Int4"
!sleep 30
!ls -lh /content/steady_decode*

-rw-rw-r-- 1 root root 0 Sep 20 16:20 /content/steady_decode.1.nsys-rep


In [ ]:
!nsys stats \
  --report cuda_gpu_kern_sum \
  --report cuda_api_sum \
  /content/steady_decode.1.nsys-rep

Generating SQLite file /content/steady_decode.1.sqlite from /content/steady_decode.1.nsys-rep
Processing [/content/steady_decode.1.sqlite] with [/opt/nvidia/nsight-systems-cli/2026.5.1/target-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                                                  Name                                                
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  ----------------------------------------------------------------------------------------------------
     28.3      369,308,470        256  1,442,611.2  1,383,709.0  1,378,109  2,276,134    215,245.5  std::enable_if<T7, void>::type internal::gemvx::kernel<int, int, __half, __half, __half, float, (bo…
     21.9      285,898,632     14,280     20,020.9     20,319.0     14,432     42,495      5,123.5  void marli

## 6. Steady-State Nsight Systems Findings

After isolating steady-state decode, the dominant GPU work is substantially clearer.

The main kernel families are approximately:

| Kernel family | Approx. GPU kernel time |
|---|---:|
| `gemvx` | ~28.5% |
| Marlin variant #1 | ~21.9% |
| Marlin variant #2 | ~17.3% |
| `kernel_unified_attention` | ~14.7% |

The two dominant Marlin variants together account for roughly **39% of GPU kernel time**, making the quantized linear path a major optimization target.

The launch counts also match the transformer decode structure. For example, the attention kernel appears 7,168 times:

`7,168 / 256 generated tokens = 28 launches per token`

which is consistent with the model's 28 transformer layers.

At this stage the profiler tells us **where time is being spent**, but not yet **why the Marlin kernels are expensive**.

That requires Nsight Compute.

### 6.1 Exploratory Attempt: Mapping Kernels Back to Model Layers

Before moving to hardware-counter analysis, I briefly tested vLLM's layerwise NVTX tracing to determine whether individual Marlin/GEMM invocations could be mapped cleanly back to model-level operations.

This run uses `--enforce-eager` and enables layerwise NVTX annotations.

**Important:** this configuration changes the normal optimized execution path by disabling CUDA Graphs and compilation optimizations. It is therefore used only for attribution/debugging and not for performance measurement.

In [ ]:

!nohup nsys profile \
  --trace=cuda,nvtx,osrt \
  --trace-fork-before-exec=true \
  --force-overwrite=true \
  -o /content/nvtx_mapping \
  vllm serve JunHowie/Qwen3-0.6B-GPTQ-Int4 \
    --port 8000 \
    --enforce-eager \
    --enable-layerwise-nvtx-tracing \
  > /content/nvtx_server.log 2>&1 &

In [ ]:
import time, requests

for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8000/health").status_code == 200:
            print("server ready")
            break
    except:
        !tail -2 /content/nvtx_server.log
        pass
    time.sleep(2)

server ready


In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 16 \
  --num-prompts 1 \
  --ignore-eos \
  --temperature 0 \
  --port 8000


# !vllm bench serve \
#   --backend vllm \
#   --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
#   --dataset-name random \
#   --input-len 32 \
#   --output-len 256 \
#   --num-prompts 1 \
#   --ignore-eos \
#   --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7923ec1a8d60>, trust_remote_code=False, seed=0, num_prompts=1, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=

In [ ]:
!pkill -INT -f "vllm serve"
!sleep 30


In [ ]:
!ls -lh /content/nvtx_mapping*

-rw-rw-r-- 1 root root 0 Sep 20 16:48 /content/nvtx_mapping.nsys-rep


In [ ]:
!nsys stats \
  --report nvtx_sum \
  /content/nvtx_mapping.nsys-rep | head -100

Generating SQLite file /content/nvtx_mapping.sqlite from /content/nvtx_mapping.nsys-rep
Processing [/content/nvtx_mapping.sqlite] with [/opt/nvidia/nsight-systems-cli/2026.5.1/target-linux-x64/reports/nvtx_sum.py]... 

 ** NVTX Range Summary (nvtx_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)    Med (ns)   Min (ns)   Max (ns)   StdDev (ns)   Style                    Range                 
 --------  ---------------  ---------  -----------  ---------  --------  ----------  -----------  -------  ---------------------------------------
     97.2       53,215,990         19  2,800,841.6  175,918.0   152,334  32,271,076  8,201,730.4  PushPop  CCCL:cub::DeviceRadixSort              
      1.8          976,401         19     51,389.5   34,819.0    26,206     182,339     40,602.7  PushPop  CCCL:cub::DeviceScan::InclusiveScan    
      1.0          530,790         10     53,079.0   37,035.5    23,872     172,857     45,365.4  PushPop  CCCL:cub::DeviceScan::InclusiveSumByKey



### NVTX Attribution Result

The NVTX summary primarily exposed internal CUB ranges and did not provide useful attribution of the dominant Marlin kernels to specific model layers.

Because this experiment also requires eager execution and changes the optimized runtime path, further NVTX investigation was deprioritized.

The important system-level findings remain:

- Marlin kernels: ~39% of GPU kernel time
- `gemvx`: ~28.5%
- attention: ~14.7%

The next step is therefore to profile Marlin directly with Nsight Compute and determine whether it is limited by compute throughput, memory bandwidth, occupancy, or scheduler stalls.

## 7. Kernel-Level Analysis with Nsight Compute

Nsight Systems established that Marlin is a major component of steady-state GPTQ decode.

The next question is:

> **Why is the Marlin kernel expensive on this workload?**

Nsight Compute is used to inspect:

- compute utilization,
- memory and DRAM utilization,
- register usage,
- shared-memory usage,
- achieved occupancy,
- scheduler eligibility,
- and warp behavior.

The first NCU experiment captures one Marlin invocation with the full metric set.

In [ ]:
!ncu \
  --target-processes all \
  --set full \
  --kernel-name Marlin \
  --launch-count 1 \
  --force-overwrite \
  -o /content/marlin_ncu \
  vllm bench latency \
    --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
    --input-len 32 \
    --output-len 32 \
    --batch-size 1 \
    --num-iters-warmup 2 \
    --num-iters 1

INFO 09-20 17:10:51 [api_utils.py:286] non-default args: {'enable_prefix_caching': False, 'mm_device_do_normalize': None, 'enable_lora': None, 'reasoning_parser_plugin': '', 'enable_bf16x3_router_gemm': False, 'model': 'JunHowie/Qwen3-0.6B-GPTQ-Int4'}
INFO 09-20 17:10:52 [model.py:684] Resolved architecture: Qwen3ForCausalLM
WARNING 09-20 17:10:52 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-20 17:10:52 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 17:10:52 [model.py:2021] Using max model len 40960
INFO 09-20 17:10:53 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 17:10:54 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=22870) INFO 09-20 17:11:17 [core.py:123] Initializing a V1 LLM engine (v0.29.0) w

### 7.1 First NCU Capture: Useful, but Not Representative of Steady-State Decode

The first successful Nsight Compute capture showed significant resource pressure, including high register usage, large shared-memory allocation, and low occupancy.

However, the captured Marlin invocation had a duration of several milliseconds, while the steady-state Nsight Systems trace showed dominant Marlin kernels operating on the order of tens of microseconds.

This indicates that the first NCU capture selected a Marlin invocation from initialization or warm-up rather than the representative steady-state decode path.

I therefore do **not** use this first capture as the final bottleneck characterization.

A later Marlin invocation must be sampled.

In [ ]:
# !sleep 30
!ncu --import /content/marlin_ncu.ncu-rep --page details

[22870] python3.13@127.0.0.1
  void Marlin<1125899906910725, 1125899907892224, 1125899906910725, 1125899906910725, 256, 4, 16, 4, 0, 2, 8, 0>(const int4 *, const int4 *, int4 *, int4 *, const int4 *, const float *, const int4 *, const float *, const int4 *, const int *, int, int, int, int, int, int *, bool, bool, bool, int) (40, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.01
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle    3,135,875
    Memory Throughput                 %        20.14
    DRAM Throughput                   %         8.71
    Duration                         ms         5.36
    L1/TEX Cache Throughput           %        40.29
    L2 Cache Throughput               %

## 8. Capturing a Representative Steady-State Marlin Kernel

To avoid profiling an initialization/warm-up invocation, Nsight Compute skips the first several thousand matching Marlin launches and captures one later invocation.

Only the sections needed for bottleneck classification are collected:

- `SpeedOfLight`
- `LaunchStats`
- `Occupancy`
- `SchedulerStats`
- `WarpStateStats`

`--enforce-eager` is used here to simplify kernel-level NCU capture and avoid CUDA Graph complications.

Because eager mode changes normal vLLM execution, **end-to-end latency from this run is not used as a benchmark**. The run is used only to inspect the hardware behavior of a representative Marlin kernel.

In [ ]:
!ncu \
  --target-processes all \
  --kernel-name Marlin \
  --launch-skip 8000 \
  --launch-count 1 \
  --section SpeedOfLight \
  --section LaunchStats \
  --section Occupancy \
  --section SchedulerStats \
  --section WarpStateStats \
  --kill yes \
  --force-overwrite \
  -o /content/marlin_steady_min \
  vllm bench latency \
    --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
    --input-len 32 \
    --output-len 64 \
    --batch-size 1 \
    --num-iters-warmup 2 \
    --num-iters 1 \
    --enforce-eager

INFO 09-20 17:51:56 [api_utils.py:286] non-default args: {'enable_prefix_caching': False, 'enforce_eager': True, 'mm_device_do_normalize': None, 'enable_lora': None, 'reasoning_parser_plugin': '', 'enable_bf16x3_router_gemm': False, 'model': 'JunHowie/Qwen3-0.6B-GPTQ-Int4'}
INFO 09-20 17:51:57 [model.py:684] Resolved architecture: Qwen3ForCausalLM
WARNING 09-20 17:51:57 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-20 17:51:57 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 17:51:57 [model.py:2021] Using max model len 40960
INFO 09-20 17:51:58 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 09-20 17:51:59 [vllm.py:1371] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-20 17:51:59 [vllm.py:1406] Inductor compilation was disable

In [ ]:
!ncu --import /content/marlin_steady_min.ncu-rep --page details

[34032] python3.13@127.0.0.1
  void Marlin<1125899906910725, 1125899907892224, 1125899906910725, 1125899906910725, 256, 2, 16, 4, 0, 2, 8, 0>(const int4 *, const int4 *, int4 *, int4 *, const int4 *, const float *, const int4 *, const float *, const int4 *, const int *, int, int, int, int, int, int *, bool, bool, bool, int) (40, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.71
    Elapsed Cycles                cycle       32,898
    Memory Throughput                 %        14.49
    DRAM Throughput                   %        14.49
    Duration                         us        56.26
    L1/TEX Cache Throughput           %        23.04
    L2 Cache Throughput               %

## 9. Confirmed Marlin Bottleneck

The later Marlin invocation shows a different and much more representative steady-state execution profile.

### Key Metrics

| Metric | Result |
|---|---:|
| Compute (SM) throughput | 28.09% |
| Memory throughput | 14.49% |
| Registers per thread | 198 |
| Dynamic shared memory per block | 65.54 KB |
| Theoretical occupancy | 25% |
| Achieved occupancy | 26.06% |
| Active warps per scheduler | 2.00 |
| Eligible warps per scheduler | 0.26 |
| Scheduler cycles with no eligible warp | 82.01% |

### Interpretation

The kernel is **not saturating compute throughput** and is also **not saturating DRAM bandwidth**.

Instead, its resource footprint limits residency:

- 198 registers per thread,
- ~65.5 KB shared memory per block,
- approximately one resident block per SM,
- and only ~25% theoretical occupancy.

With relatively few resident warps, the scheduler frequently has no warp ready to execute. Nsight Compute reports `No Eligible = 82.01%`.

The resulting bottleneck is best characterized as:

> **low residency / poor latency hiding rather than simple compute or DRAM-bandwidth saturation.**

A simplified causal chain is:

**register + shared-memory pressure → limited block residency → low occupancy → few eligible warps → poor latency hiding → underutilized compute and memory resources**

This completes the primary profiling and bottleneck-diagnosis phase.

# Exploratory Optimization Experiment — Marlin Atomic Add

> **Status: investigated and not adopted**

After diagnosing the Marlin execution path, I evaluated one readily available alternative Marlin configuration: `VLLM_MARLIN_USE_ATOMIC_ADD=1`.

This was an exploratory optimization candidate rather than the final optimization direction.

The purpose of the experiment was to test whether changing Marlin's reduction behavior could improve low-batch decode performance.

A controlled A/B comparison is performed with all workload parameters held constant:

- same model,
- same GPU,
- input length = 32,
- output length = 256,
- concurrency = 1,
- 20 measured requests,
- 3 warm-up requests.

The only intended configuration difference is the Marlin atomic-add setting.

## A. Baseline Configuration

The baseline runs the default Marlin execution path with:

`VLLM_MARLIN_USE_ATOMIC_ADD` unset.

This provides the reference end-to-end decode measurements for the experimental comparison.

In [ ]:
!pkill -INT -f "vllm serve" || true
!mkdir -p /content/marlin_ab

In [ ]:
%%bash
unset VLLM_MARLIN_USE_ATOMIC_ADD

nohup vllm serve JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --port 8000 \
  > /content/marlin_ab/baseline_server.log 2>&1 &

echo $!

37472


In [ ]:
import time, requests

for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8000/health").status_code == 200:
            print("ready")
            break
    except:
        pass
    time.sleep(1)

ready


In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 256 \
  --num-prompts 20 \
  --num-warmups 3 \
  --max-concurrency 1 \
  --ignore-eos \
  --temperature 0 \
  --save-result \
  --result-dir /content/marlin_ab \
  --result-filename baseline.json \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7bfd94ec8d60>, trust_remote_code=False, seed=0, num_prompts=20, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size

In [ ]:
!pkill -INT -f "vllm serve" || true
!sleep 5

## B. Atomic-Add Configuration

The same workload is repeated with:

`VLLM_MARLIN_USE_ATOMIC_ADD=1`

No other benchmark parameter is intentionally changed.

The primary metrics of interest are **TPOT** and **output-token throughput**, since this experiment targets autoregressive decode rather than prompt processing.

In [ ]:
%%bash
export VLLM_MARLIN_USE_ATOMIC_ADD=1

nohup vllm serve JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --port 8000 \
  > /content/marlin_ab/atomic_server.log 2>&1 &

echo $!

38288


In [ ]:
import time, requests

for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8000/health").status_code == 200:
            print("ready")
            break
    except:
        pass
    time.sleep(1)

ready


In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 256 \
  --num-prompts 20 \
  --num-warmups 3 \
  --max-concurrency 1 \
  --ignore-eos \
  --temperature 0 \
  --save-result \
  --result-dir /content/marlin_ab \
  --result-filename atomic.json \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7afb85bcccc0>, trust_remote_code=False, seed=0, num_prompts=20, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size

### End-to-End A/B Result

Initial measurements:

| Metric | Baseline | Atomic Add |
|---|---:|---:|
| Mean TPOT | 4.61 ms | 4.88 ms |
| Mean ITL | 4.61 ms | 4.88 ms |
| Output throughput | 212.19 tok/s | 201.06 tok/s |
| Mean TTFT | 29.62 ms | 27.87 ms |

For the decode-focused metrics, atomic-add did **not** improve performance:

- TPOT increased by approximately 5.9%.
- Output throughput decreased by approximately 5.2%.

Before rejecting the configuration, I use Nsight Systems to check whether the kernel execution behavior actually changed.

In [ ]:
!pkill -INT -f "vllm serve" || true
!sleep 5

## C. Kernel-Level Validation of the Experimental Configuration

End-to-end measurements alone do not explain what changed internally.

I therefore capture one steady-state decode request for both configurations using Nsight Systems and compare:

- Marlin kernel time,
- `reduce_segments`,
- the dominant unrelated kernels,
- and kernel invocation counts.

The goal is to determine whether any apparent Marlin improvement can be causally attributed to the atomic-add configuration.

In [ ]:
%%bash

unset VLLM_MARLIN_USE_ATOMIC_ADD

nohup nsys profile \
  --trace=cuda,nvtx,osrt \
  --trace-fork-before-exec=true \
  --cuda-graph-trace=node \
  --capture-range=cudaProfilerApi \
  --capture-range-end=repeat \
  --force-overwrite=true \
  -o /content/marlin_ab/nsys_baseline \
  vllm serve JunHowie/Qwen3-0.6B-GPTQ-Int4 \
    --port 8000 \
    --profiler-config.profiler cuda \
  > /content/marlin_ab/nsys_baseline_server.log 2>&1 &

In [ ]:
import time, requests

for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8000/health").status_code == 200:
            print("ready")
            break
    except:
        pass
    time.sleep(1)

ready


In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 32 \
  --num-prompts 1 \
  --ignore-eos \
  --temperature 0 \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x786aac940e00>, trust_remote_code=False, seed=0, num_prompts=1, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=

In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 256 \
  --num-prompts 1 \
  --ignore-eos \
  --temperature 0 \
  --profile \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7bfc141b1580>, trust_remote_code=False, seed=0, num_prompts=1, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=

In [ ]:
!pkill -INT -f "vllm serve" || true
!sleep 5

In [ ]:
!sleep 30
!nsys stats \
  --report cuda_gpu_kern_sum \
  /content/marlin_ab/nsys_baseline.2.nsys-rep \
  > /content/marlin_ab/baseline_kernel_stats.txt

#### REPEAT BUT WITH export VLLM_MARLIN_USE_ATOMIC_ADD=1

In [ ]:
%%bash

export VLLM_MARLIN_USE_ATOMIC_ADD=1


nohup nsys profile \
  --trace=cuda,nvtx,osrt \
  --trace-fork-before-exec=true \
  --cuda-graph-trace=node \
  --capture-range=cudaProfilerApi \
  --capture-range-end=repeat \
  --force-overwrite=true \
  -o /content/marlin_ab/nsys_atomic \
  vllm serve JunHowie/Qwen3-0.6B-GPTQ-Int4 \
    --port 8000 \
    --profiler-config.profiler cuda \
  > /content/marlin_ab/nsys_atomic_server.log 2>&1 &

In [ ]:
import time, requests

for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8000/health").status_code == 200:
            print("ready")
            break
    except:
        pass
    time.sleep(1)

ready


In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 32 \
  --num-prompts 1 \
  --ignore-eos \
  --temperature 0 \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7830fa3f4e00>, trust_remote_code=False, seed=0, num_prompts=1, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=

In [ ]:
!vllm bench serve \
  --backend vllm \
  --model JunHowie/Qwen3-0.6B-GPTQ-Int4 \
  --dataset-name random \
  --input-len 32 \
  --output-len 256 \
  --num-prompts 1 \
  --ignore-eos \
  --temperature 0 \
  --profile \
  --port 8000

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7f9460e04d60>, trust_remote_code=False, seed=0, num_prompts=1, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=

In [ ]:
!pkill -INT -f "vllm serve" || true
!sleep 5

In [ ]:
!sleep 5
!nsys stats \
  --report cuda_gpu_kern_sum \
  /content/marlin_ab/nsys_atomic.1.nsys-rep \
  > /content/marlin_ab/atomic_kernel_stats.txt

#### compare

In [ ]:
!echo "baseline"
!grep -E "marlin::Marlin|reduce_segments" \
  /content/marlin_ab/baseline_kernel_stats.txt

baseline
     23.3      303,827,935     14,280     21,276.5     21,727.0     14,464     40,479      5,193.1  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
     18.8      244,906,101     14,280     17,150.3     16,768.0     11,872     32,319      3,953.7  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
      2.1       26,764,301      7,140      3,748.5      3,552.0      2,847      6,432        732.6  reduce_segments                                                                                     
      0.2        2,488,604         56     44,439.4     44,319.0     38,655     50,238      5,222.8  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
      0.1        1,879,305         56     33,559.0     33,871.0     28,448     38,111      3,937.7  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910

In [ ]:
!echo "baseline"
!grep -E "marlin::Marlin|reduce_segments" \
  /content/marlin_ab/baseline_kernel_stats.txt

baseline
     23.2      294,571,648     14,280     20,628.3     20,607.0     14,431     40,543      5,816.7  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
     18.6      236,241,110     14,280     16,543.5     15,680.0     11,808     32,159      4,541.4  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
      2.0       25,783,334      7,140      3,611.1      3,264.0      2,784      6,400        885.1  reduce_segments                                                                                     
      0.2        2,489,092         56     44,448.1     44,430.5     38,655     50,335      5,223.1  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
      0.1        1,884,595         56     33,653.5     33,695.0     28,831     38,111      3,875.3  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910

In [ ]:
!echo "atomic"
!grep -E "marlin::Marlin|reduce_segments" \
  /content/marlin_ab/atomic_kernel_stats.txt

atomic
     23.0      277,657,123     14,280     19,443.8     19,744.0     14,272     40,575      5,598.9  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
     18.2      219,809,713     14,280     15,392.8     14,687.0     11,679     32,415      4,335.2  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
      2.0       23,913,513      7,140      3,349.2      3,103.0      2,752      6,495        836.6  reduce_segments                                                                                     
      0.2        2,490,691         56     44,476.6     44,222.5     38,431     50,271      5,149.0  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)1125899906910725, (long)1…
      0.2        1,874,964         56     33,481.5     33,567.5     28,767     38,751      3,991.2  void marlin::Marlin<(long)1125899906910725, (long)1125899907892224, (long)112589990691072

In [ ]:
!echo "=== BASELINE ==="
!head -30 /content/marlin_ab/baseline_kernel_stats.txt

!echo "=== ATOMIC ==="
!head -30 /content/marlin_ab/atomic_kernel_stats.txt

=== BASELINE ===
Generating SQLite file /content/marlin_ab/nsys_baseline.2.sqlite from /content/marlin_ab/nsys_baseline.2.nsys-rep
Processing [/content/marlin_ab/nsys_baseline.2.sqlite] with [/opt/nvidia/nsight-systems-cli/2026.5.1/target-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                                                  Name                                                
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  ----------------------------------------------------------------------------------------------------
     29.8      379,070,378        256  1,480,743.7  1,384,766.0  1,377,982  2,275,783    250,598.4  std::enable_if<T7, void>::type internal::gemvx::kernel<int, int, __half, __half, __half, float, (bo…
     23.2      294,571,648     14,280     20,628.3     20,607.0

## D. Interpretation: Atomic Add Is Not a Defensible Optimization Result

The atomic trace showed lower durations for the dominant Marlin kernels.

However, the same trace also showed similar improvements in unrelated kernels, including attention and `gemvx`.

For example, both Marlin and attention became roughly 6–7% faster in the same trace, even though the Marlin atomic-add setting should not directly optimize the attention kernel.

This suggests that the raw difference between the two Nsight traces is affected by run-to-run factors such as GPU clock state, power/thermal behavior, or general runtime variability.

More importantly, the controlled end-to-end benchmark showed worse decode TPOT with atomic-add enabled.

Therefore I do **not** claim an atomic-add speedup from these measurements.

### Conclusion of This Experiment

**Atomic-add was tested and rejected as the optimization direction for this workload.**

This is an example of why kernel-level observations must be validated against end-to-end application performance before an optimization is accepted.

In [ ]:
!ls -alhtr /content/marlin_ab


total 72M
drwxr-xr-x 1 root root 4.0K Sep 20 18:03 ..
-rw-r--r-- 1 root root 1.1K Sep 20 18:06 baseline.json
-rw-r--r-- 1 root root  28K Sep 20 18:07 baseline_server.log
-rw-r--r-- 1 root root 1.1K Sep 20 18:10 atomic.json
-rw-r--r-- 1 root root  30K Sep 20 18:13 atomic_server.log
-rw-rw-r-- 1 root root 5.9M Sep 20 18:16 nsys_baseline.1.nsys-rep
-rw-r--r-- 1 root root  19M Sep 20 18:17 nsys_baseline.1.sqlite
-rw-rw-r-- 1 root root 5.8M Sep 20 18:24 nsys_atomic.1.nsys-rep
-rw-r--r-- 1 root root  35K Sep 20 18:24 nsys_atomic_server.log
-rw-r--r-- 1 root root  18M Sep 20 18:24 nsys_atomic.1.sqlite
-rw-r--r-- 1 root root 7.1K Sep 20 18:24 atomic_kernel_stats.txt
-rw-rw-r-- 1 root root 5.9M Sep 20 18:38 nsys_baseline.2.nsys-rep
-rw-r--r-- 1 root root  30K Sep 20 18:38 nsys_baseline_server.log
drwxr-xr-x 2 root root 4.0K Sep 20 18:38 .
-rw-r--r-- 1 root root  19M Sep 20 18:38 nsys_baseline.2.sqlite
-rw-r--r-- 1 root root 7.1K Sep 20 18:38 baseline_kernel_stats.txt


# Current Project Status

## Completed

The profiling phase established the following:

1. vLLM executes the GPTQ INT4 model through Marlin.
2. Marlin kernels account for a substantial fraction of steady-state GPU decode time.
3. A representative Marlin decode kernel does not saturate either compute or DRAM bandwidth.
4. Register and shared-memory requirements limit block residency.
5. Achieved occupancy is approximately 26%.
6. The scheduler has no eligible warp during approximately 82% of cycles.
7. The observed bottleneck is therefore consistent with limited latency hiding rather than simple bandwidth saturation.

## Optimization Status

One experimental Marlin configuration (`VLLM_MARLIN_USE_ATOMIC_ADD=1`) was evaluated but did not produce a defensible end-to-end improvement and was abandoned.

The optimization stage remains **open**.

Potential next directions include:

- investigating Marlin tile/resource configurations,
- evaluating another quantized GEMM execution path,
- reducing register/shared-memory pressure,
- or implementing a small W4A16 kernel experiment for one representative model shape.

The next optimization will be selected based on the profiler evidence rather than chosen solely from benchmark results.

# Appendix — Saving Profiler Artifacts

The following cell copies generated Nsight reports and logs to persistent storage.

This is an artifact-management step and is not part of the profiling methodology or performance analysis.

In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/gdrive')

# Define source and destination paths
source_dir = '/content'
dest_dir = '/content/gdrive/MyDrive/gpu_bench'

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# List of items to ignore (folders or files) in the source directory
# 'gdrive' is ignored to prevent attempts to copy the mounted drive itself
ignore_items = ['.config', 'sample_data', 'gdrive', 'usr', 'opt', 'bin', 'etc', 'lib', 'lib32', 'lib64', 'media', 'mnt', 'proc', 'run', 'srv', 'sys', 'tmp', 'var', 'root', 'dev', 'home', 'sbin', 'snap', 'boot', 'dataroot', 'tools', 'datalab', 'kaggle', 'output', 'tensorflow-workspace']


# File extensions to copy
extensions_to_copy = ('.sqlite', '.nsys-rep', '.log','.ncu-rep')

print(f"Copying files with extensions {extensions_to_copy} from {source_dir} to {dest_dir} (ignoring {ignore_items} and already existing files)...")

# Recursively walk through the source directory
for root, dirs, files in os.walk(source_dir):
    # Modify dirs in-place to skip ignored directories
    dirs[:] = [d for d in dirs if d not in ignore_items]

    # Skip root directory if it's in ignore_items (e.g., '/content/.config')
    # This handles top-level directories that os.walk would still list
    if os.path.basename(root) in ignore_items and root != source_dir:
        print(f"  Skipping ignored directory: {root}/")
        continue

    for file_name in files:
        source_file_path = os.path.join(root, file_name)
        # Construct the destination path, preserving relative directory structure
        relative_path = os.path.relpath(source_file_path, source_dir)
        dest_file_path = os.path.join(dest_dir, relative_path)

        if file_name.endswith(extensions_to_copy):
            # Ensure the destination subdirectory exists
            os.makedirs(os.path.dirname(dest_file_path), exist_ok=True)

            if os.path.exists(dest_file_path):
                print(f"  Skipping already existing file: {relative_path}")
            else:
                try:
                    shutil.copy2(source_file_path, dest_file_path)
                    print(f"  Copied file: {relative_path}")
                except Exception as e:
                    print(f"  Error copying file {relative_path}: {e}")
        else:
            print(f"  Skipping file (does not match specified extensions): {relative_path}")

print("Copying complete.")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Copying files with extensions ('.sqlite', '.nsys-rep', '.log', '.ncu-rep') from /content to /content/gdrive/MyDrive/gpu_bench (ignoring ['.config', 'sample_data', 'gdrive', 'usr', 'opt', 'bin', 'etc', 'lib', 'lib32', 'lib64', 'media', 'mnt', 'proc', 'run', 'srv', 'sys', 'tmp', 'var', 'root', 'dev', 'home', 'sbin', 'snap', 'boot', 'dataroot', 'tools', 'datalab', 'kaggle', 'output', 'tensorflow-workspace'] and already existing files)...
  Skipping already existing file: vllm_server.log
  Skipping already existing file: nvtx_server.log
  Skipping already existing file: nvtx_mapping.nsys-rep
  Skipping already existing file: steady_decode.1.nsys-rep
  Copied file: marlin_steady_server.log
  Skipping already existing file: decode_profile.sqlite
  Skipping already existing file: nvtx_mapping.sqlite
  Skipping already existing file: decode_profile.nsys-rep
  Skipp